# Fine-tuning du modèle de texte (Google Colab)

Ce notebook ne contient pas de logique : il lance `complaint_triage.models.transformer`, dont le code est versionné et relu dans le dépôt.

1. Activer un GPU : *Exécution > Modifier le type d'exécution > GPU T4*.
2. Déposer `recits-colab.parquet` (produit par `scripts/experiments/export_text_for_colab.py`) dans le dossier `complaint-triage` de Google Drive.
3. Exécuter les cellules dans l'ordre.
4. Récupérer `scores-validation-distilroberta.parquet` dans le même dossier Drive et le copier dans `data/processed/`.

Le fichier ne contient pas la période de test, qui reste sur la machine locale.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Activer un GPU : Exécution > Modifier le type d'exécution")
print(torch.cuda.get_device_name(0))

In [ ]:
# --no-deps : Colab fournit déjà pandas, scikit-learn, PyTorch et Transformers.
!pip install -q --no-deps "git+https://github.com/mangnathan/complaint-triage@main"
!pip install -q datasets accelerate

In [ ]:
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
FOLDER = "/content/drive/MyDrive/complaint-triage"
narratives = pd.read_parquet(f"{FOLDER}/recits-colab.parquet")
narratives["partie"].value_counts()

In [ ]:
from complaint_triage.models.transformer import TransformerConfig, fine_tune_and_score

config = TransformerConfig()
scores = fine_tune_and_score(
    fit=narratives[narratives["partie"] == "apprentissage"],
    stop=narratives[narratives["partie"] == "arret"],
    to_score=narratives[narratives["partie"] == "validation"],
    output_dir="/content/entrainement",
    config=config,
)
scores.to_parquet(f"{FOLDER}/scores-validation-distilroberta.parquet", index=False)
print(config)
print(len(scores), "réclamations notées")